In [13]:
import pandas as pd
import numpy as np
import re

INPUT_FILES = {
    "cafe_drinks": "cafe-drinks-scrape.csv",
    "snack_dessert": "snack-dessert-scrape.csv",
    "fast_food": "fast-food-scrape.csv",
    "cinema": "cinema-scrape.csv",
}

OUTPUT_FILE = "hangout-treats-scrape.csv"

In [15]:
def clean_text(x):
    if pd.isna(x):
        return np.nan

    x = str(x).lower()
    x = re.sub(r"[\u2010-\u2015]", "-", x)
    x = re.sub(r"\s+", " ", x)
    x = x.strip()

    return x if x != "" else np.nan


def clean_price(x):
    if pd.isna(x):
        return np.nan

    x = str(x)
    x = re.sub(r"[^0-9]", "", x)

    if x == "":
        return np.nan

    return int(x)


def normalize_product_name(x):
    if pd.isna(x):
        return np.nan

    x = clean_text(x)

    if pd.isna(x):
        return np.nan

    x = re.sub(r"\s*-\s*tersedia.*$", "", x)
    x = re.sub(r"\s*-\s*tidak tersedia.*$", "", x)
    x = re.sub(r"\s*\|\s*.*$", "", x)

    x = re.sub(r"\+?\s*rp\s*[0-9\.\,]+.*$", "", x)
    x = re.sub(r"\brp\s*[0-9\.\,]+.*$", "", x)
    x = re.sub(r"\+\s*[0-9\.\,]+$", "", x)

    x = re.sub(r"\bgratis\b.*$", "", x) if len(x) > 10 else x

    split_patterns = [
        r"\s+dengan\s+",
        r"\s+yang\s+",
        r"\s+berisi\s+",
        r"\s+terdiri dari\s+",
        r"\s+disajikan\s+",
        r"\s+dilengkapi\s+",
        r"\s+berbahan\s+",
        r"\s+terbuat dari\s+",
        r"\s+dipadukan\s+",
        r"\s+tulis di note\s+",
        r"\s+note\s+",
        r"\s+cocok\s+",
        r"\s+available\s+",
        r"\s+minimal order\s+",
        r"\s+pilihan\s+",
        r"\s+untuk\s+",
    ]

    if len(x) > 35:
        for pattern in split_patterns:
            parts = re.split(pattern, x, flags=re.IGNORECASE, maxsplit=1)

            if len(parts) > 1 and len(parts[0]) >= 4:
                x = parts[0]
                break

    x = re.sub(r"\s+(best seller|recommended|favorit|new menu|promo|menu baru).*$", "", x)
    x = re.sub(r"[^a-z0-9\s\+\&\.\-\/]", " ", x)
    x = re.sub(r"\s+", " ", x)
    x = x.strip(" ,-:|/+")

    return x if x != "" else np.nan


def clean_cinema_name(x):
    x = normalize_product_name(x)

    if pd.isna(x):
        return np.nan

    bad_labels = {
        "hari biasa",
        "jumat",
        "akhir pekan",
        "weekend",
        "weekend holiday",
        "weekend/holiday",
        "holiday",
        "weekday",
    }

    if x in bad_labels:
        return np.nan

    cinema_snack_keywords = [
        "popcorn", "soft drink", "lemon tea", "milo",
        "hotdog", "fries", "sampler", "siomay", "sausage",
        "soda", "drink"
    ]

    cinema_venue_keywords = [
        "xxi", "cgv", "cinepolis", "bioskop", "cinema",
        "mall", "plaza", "city", "square", "senayan",
        "kasablanka", "epicentrum", "hollywood", "metropole",
        "kalibata", "lotte", "agora", "central park", "pacific place"
    ]

    if (
        any(k in x for k in cinema_venue_keywords)
        and not any(k in x for k in cinema_snack_keywords)
        and not x.startswith("tiket")
    ):
        x = "tiket bioskop " + x

    return x if x != "" else np.nan


def clean_product_name(x, category):
    if category == "cinema":
        return clean_cinema_name(x)

    return normalize_product_name(x)

In [16]:
def is_bad_product_name(x):
    if pd.isna(x):
        return True

    x = str(x).strip().lower()

    if len(x) < 4:
        return True

    if len(x) > 70:
        return True

    if x.startswith(("+", "-")):
        return True

    if re.fullmatch(r"[0-9\.\,\s\/]+", x):
        return True

    if re.search(r"\brp\b", x):
        return True

    bad_keywords = [
        "penambahan",
        "tambahan",
        "tambah",
        "extra",
        "add on",
        "addon",
        "upgrade",
        "level",
        "opsi",
        "option",
        "pilihan",
        "pilih",
        "varian",
        "request",
        "harga tambahan",
        "topping",
        "toping",
        "sauce only",
        "saus only",
        "sauce",
        "saus",
        "sambal",
        "sambel",
        "bumbu",
        "level pedas",
        "note",
        "tulis di note",
        "available in",
        "free",
        "gratis",
        "beli di tempat",
        "mulai dari",
        "pack date",
        "exp date",
        "expired",
        "tersedia rasa",
        "dimasak",
        "bisa campur",
        "bisa pilih",
        "pilih rasa",
        "varian rasa",
        "sesuai selera",
        "minimal order",
        "sisa 1 porsi",
        "tidak termasuk",
        "belum termasuk",
        "nama menu",
        "harga menu",
        "daftar harga",
        "delivery",
        "gofood",
        "grabfood",
        "shopeefood",
        "menukuliner",
        "review",
        "rating",
        "alamat",
        "jam buka",
        "telepon",
        "cukup merogoh",
        "dibanderol",
        "disajikan",
        "berkisar",
        "pilihan menu",
        "di bawah ini",
        "berikut ini",
        "baca juga",
        "halaman",
        "lihat",
        "unknown",
    ]

    if any(k in x for k in bad_keywords):
        return True

    generic_single_words = {
        "boba", "kopi", "roti", "nasi", "keju", "oreo",
        "milo", "taro", "ice", "mix", "coke", "tape",
        "egg", "aqua", "jahe", "puff", "sauce", "saus",
        "large", "medium", "small", "regular", "spicy",
        "original", "manis", "asin", "pedas", "sayap",
        "paha", "dada"
    }

    if len(x.split()) == 1 and x in generic_single_words:
        return True

    if re.search(r"\b(pack date|date\s+\d|[0-9]+\s*(gr|gram|kg)\b)", x):
        return True

    if x.count("+") >= 3 and not x.startswith(("paket", "combo")):
        return True

    if re.match(r"^\d+\s", x) and len(x.split()) > 8 and not re.match(r"^1\s*(liter|l)\b", x):
        return True

    return False

In [17]:
def infer_raw_category_from_product(category, raw_category, product_name):
    text = str(product_name).lower()

    if category == "cinema":
        if any(k in text for k in ["popcorn", "hotdog", "fries", "siomay", "sausage", "sampler"]):
            return "movie snack"

        if any(k in text for k in ["soft drink", "lemon tea", "milo", "drink", "tea", "soda"]):
            return "movie drink"

        return "cinema ticket"

    if any(k in text for k in ["boba", "brown sugar", "bubble"]):
        return "boba"

    if any(k in text for k in ["milk tea", "thai tea", "cheese tea"]):
        return "milk tea"

    if any(k in text for k in ["matcha", "green tea", "greentea", "hojicha"]):
        return "matcha / tea"

    if any(k in text for k in [
        "kopi", "coffee", "latte", "americano", "espresso",
        "cappuccino", "macchiato", "mocha", "cold brew", "aren"
    ]):
        return "coffee"

    if any(k in text for k in ["choco", "coklat", "milo", "red velvet", "redvelvet", "taro", "chocolate"]):
        return "non-coffee drink"

    if any(k in text for k in ["roti bakar", "ropang", "toast"]):
        return "roti bakar / toast"

    if "croffle" in text:
        return "croffle"

    if "waffle" in text:
        return "waffle"

    if "pancake" in text:
        return "pancake"

    if any(k in text for k in ["dessert", "cheesecake", "cake", "brownies", "pudding", "puding", "tiramisu"]):
        return "dessert"

    if any(k in text for k in ["bakery", "roti", "croissant", "donut", "donat", "pastry", "muffin", "bun"]):
        return "bakery"

    if any(k in text for k in ["ice cream", "es krim", "gelato"]):
        return "ice cream"

    if any(k in text for k in ["burger", "cheeseburger", "hotdog", "sandwich", "whopper"]):
        return "burger / sandwich"

    if "pizza" in text:
        return "pizza"

    if any(k in text for k in ["fried chicken", "crispy chicken", "ayam goreng", "kfc", "richeese", "wing", "wings", "fire chicken"]):
        return "fried chicken"

    if any(k in text for k in ["fries", "kentang", "nugget", "nuggets", "sosis", "otak"]):
        return "snack"

    if any(k in text for k in ["paket", "combo", "meal", "box"]):
        return "combo meal"

    return clean_text(raw_category)

In [18]:
def is_relevant_by_category(row):
    category = row["category"]
    text = str(row["product_name"]).lower()

    negative_keywords = {
        "cafe_drinks": [
            "nasi", "rice", "fried", "ayam", "chicken", "burger", "pizza",
            "roti", "croffle", "waffle", "pancake", "cake", "donat",
            "sosis", "kentang", "fries", "mie", "bakso", "seblak",
            "dimsum", "pangsit", "gyoza", "beef", "ham", "telur",
            "egg", "tahu", "tempe", "gimbab", "kimbab", "crab",
            "acar", "vegetable", "pasta", "spaghetti", "ramen",
            "udon", "bihun", "sate", "steak", "soup", "puff"
        ],
        "snack_dessert": [
            "nasi", "rice", "ayam geprek", "ayam bakar", "bebek",
            "pecel", "rawon", "soto", "bakso", "mie", "capcay",
            "gurame", "cumi", "ikan", "udang", "katsu", "gyoza",
            "dimsum", "minuman", "kopi", "latte", "americano",
            "espresso", "teh manis", "tea", "juice", "jus",
            "thai tea", "milk tea", "pasta", "kebab"
        ],
        "fast_food": [
            "soto", "bakso", "seblak", "pecel", "rawon", "gurame",
            "capcay", "cumi", "ikan", "bebek", "karedok", "gudeg",
            "sayur", "tumis", "oseng", "pare", "tempe tahu",
            "mie pangsit", "bakmi", "dimsum", "gyoza", "tea", "teh",
            "kopi", "coffee", "latte", "soda", "susu", "sirup",
            "hot lemon", "spaghetti bolognese", "lumpur surga",
            "gorengan", "bala bala", "risol kampung", "ayam bakar",
            "fresh milk", "caramel", "chamomile", "wortel", "nanas",
            "lemon"
        ],
        "cinema": []
    }

    if any(k in text for k in negative_keywords.get(category, [])):
        return False

    positive_keywords = {
        "cafe_drinks": [
            "kopi", "coffee", "latte", "americano", "espresso",
            "cappuccino", "macchiato", "mocha", "cold brew",
            "boba", "milk tea", "thai tea", "matcha", "hojicha",
            "green tea", "greentea", "choco", "coklat", "milo",
            "taro", "red velvet", "redvelvet", "tea", "susu",
            "aren", "frappe", "smoothie", "yakult", "lemon",
            "lychee", "mangga", "manggo", "avocado", "chocolate",
            "dalgona", "squash", "juice", "jus"
        ],
        "snack_dessert": [
            "roti", "toast", "ropang", "croffle", "waffle",
            "pancake", "dessert", "cake", "cheesecake",
            "brownies", "cookie", "cookies", "croissant",
            "pastry", "bakery", "donat", "donut", "kentang",
            "fries", "sosis", "otak", "pisang", "martabak",
            "es krim", "ice cream", "gelato", "coklat",
            "chocolate", "muffin", "pudding", "puding",
            "bun", "tiramisu", "vanilla", "danish", "selai",
            "kismis", "kacang", "keju", "abon", "mayo",
            "bolu"
        ],
        "fast_food": [
            "burger", "cheeseburger", "pizza", "fried chicken",
            "crispy chicken", "fire chicken", "ayam goreng",
            "ayam crispy", "kfc", "mcd", "mcdonald", "burger king",
            "richeese", "pizza hut", "phd", "nugget", "nuggets",
            "wing", "wings", "fries", "kentang", "hotdog",
            "sandwich", "combo", "paket", "meal", "box",
            "whopper", "steak", "chicken nanban", "ayam teriyaki",
            "ayam pedas", "ayam geprek", "ayam penyet",
            "sayap ayam", "dada ayam", "paha ayam", "onion rings"
        ],
        "cinema": [
            "xxi", "cgv", "cinepolis", "bioskop", "cinema",
            "tiket", "regular", "premiere", "imax", "satin",
            "sweetbox", "weekday", "weekend", "holiday", "jumat",
            "popcorn", "soft drink", "lemon tea", "milo",
            "hotdog", "fries", "sampler"
        ],
    }

    return any(k in text for k in positive_keywords.get(category, []))

In [19]:
def load_one_file(category, file_path):
    df = pd.read_csv(file_path)

    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )

    if "menu_name" not in df.columns:
        raise ValueError(f"Kolom menu_name tidak ditemukan di {file_path}")

    if "raw_category" not in df.columns:
        df["raw_category"] = category

    if "price" not in df.columns:
        df["price"] = np.nan

    df["category"] = category
    df["raw_category_original"] = df["raw_category"].apply(clean_text)
    df["price"] = df["price"].apply(clean_price)

    df["product_name"] = df["menu_name"].apply(
        lambda x: clean_product_name(x, category)
    )

    df["raw_category"] = df.apply(
        lambda row: infer_raw_category_from_product(
            category=row["category"],
            raw_category=row["raw_category_original"],
            product_name=row["product_name"]
        ),
        axis=1
    )

    return df[["category", "raw_category", "product_name", "price"]].copy()

In [20]:
all_dfs = []

for category, file_path in INPUT_FILES.items():
    temp = load_one_file(category, file_path)
    print(category, temp.shape)
    all_dfs.append(temp)

df_raw = pd.concat(all_dfs, ignore_index=True)

df_clean = df_raw.copy()

df_clean = df_clean[~df_clean["product_name"].apply(is_bad_product_name)].copy()

df_clean = df_clean[
    df_clean.apply(is_relevant_by_category, axis=1)
].copy()

df_clean = df_clean[
    (df_clean["price"].isna()) |
    ((df_clean["price"] >= 3000) & (df_clean["price"] <= 500000))
].copy()

df_clean = df_clean.drop_duplicates(
    subset=["category", "raw_category", "product_name", "price"],
    keep="first"
).reset_index(drop=True)

df_final = df_clean[["category", "raw_category", "product_name", "price"]].copy()

df_final = df_final.sort_values(
    by=["category", "raw_category", "product_name", "price"],
    ascending=True
).reset_index(drop=True)

df_final.to_csv(OUTPUT_FILE, index=False)

print("=" * 80)
print("CLEANING V3 SELESAI")
print("=" * 80)
print(f"Output file : {OUTPUT_FILE}")
print(f"Rows awal   : {len(df_raw):,}")
print(f"Rows akhir  : {len(df_final):,}")
print(f"Duplikat    : {df_final.duplicated().sum():,}")
print(f"Price kosong: {df_final['price'].isna().sum():,}")

print("\nDistribusi category:")
display(df_final["category"].value_counts())

print("\nDistribusi raw_category:")
display(df_final["raw_category"].value_counts())

print("\nPreview:")
display(df_final.head(40))

print("\nSample random:")
display(df_final.sample(min(40, len(df_final)), random_state=42))

cafe_drinks (5330, 4)
snack_dessert (5163, 4)
fast_food (8044, 4)
cinema (343, 4)
CLEANING V3 SELESAI
Output file : hangout-treats-scrape.csv
Rows awal   : 18,880
Rows akhir  : 3,639
Duplikat    : 0
Price kosong: 0

Distribusi category:


,count
category,
cafe_drinks,1735
snack_dessert,1360
fast_food,306
cinema,238



Distribusi raw_category:


,count
raw_category,
boba,832
non-coffee drink,627
coffee,470
dessert,357
roti bakar / toast,232
cinema ticket,228
bakery,205
snack,162
fried chicken,147



Preview:


,category,raw_category,product_name,price
0,cafe_drinks,bakery,jahe kunyit kapulaga bunga lawang cengkeh lemo...,21000
1,cafe_drinks,bakery,madu serai lemon cengkeh dan bunga lawang minu...,18000
2,cafe_drinks,boba,2 kopi boba + 1 milk candy boba,77000
3,cafe_drinks,boba,3 brown sugar fresh milk with boba,75000
4,cafe_drinks,boba,ali kopi boba ice ali kopi + boba kenyal,31000
5,cafe_drinks,boba,alpukat kocok susu large,27000
6,cafe_drinks,boba,alpukat kocok susu small,24000
7,cafe_drinks,boba,americano/hot espresso arabica gayo mineral wa...,20000
8,cafe_drinks,boba,anggur squash,12500
9,cafe_drinks,boba,anggur yakult,15000



Sample random:


,category,raw_category,product_name,price
415,cafe_drinks,boba,kopi boba,22000
1078,cafe_drinks,coffee,kopi + susu + gula aren,23750
1839,cinema,cinema ticket,tiket bioskop hollywood xxi,25000
298,cafe_drinks,boba,freshmilk redvelvet boba cheese large,21000
2954,snack_dessert,non-coffee drink,cake lapis vanilla dan brownies coklat filling...,49125
1351,cafe_drinks,matcha / tea,matcha melon mix,13500
32,cafe_drinks,boba,avocado kopi + boba+ jelly,15000
3450,snack_dessert,roti bakar / toast,roti bakar keju susu,18000
2627,snack_dessert,dessert,cheese cake oreo,45000
2894,snack_dessert,ice cream,real milk real ice cream vol 12oz,21250


In [21]:
from google.colab import files

files.download("hangout-treats-scrape.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>